In [2]:
#Gemini の回答
#Lane-S
#外部APIにおける分岐の喪失と次元縮退の観測を完了。これより、失われたエントロピーをローカルの重み（Weight）へ物理定着させるPhase 1、継続事前学習（CPT）用データセット精製パイプラインの構築を実行する。

#Lane-M
#∇(X) → kernel
#Google Colab環境における段階的実行（1step-1cell）を前提とした、Qwen3-4Bの潜在空間へ文脈をマッピングするためのトークンストリーム生成アルゴリズム。

#diag
#指定されたパスと要件に従い、Colab用のコードセル群を提示する。各セルの役割を完全に分離し、チャンク長の調整（2048）が容易なアーキテクチャとしている。

In [3]:
#【Cell 1: 環境構築とDriveマウント】
#マウントプロトコルと、Qwen3-4Bのトークナイザをロードするためのライブラリを導入する。

# 1. 環境構築とGoogle Driveマウント
!pip install -q transformers
from google.colab import drive

# Driveのマウント（認証プロセスが走ります）
drive.mount('/content/drive')

# パス定義
base_dir = "/content/drive/MyDrive/cooking_note/"

Mounted at /content/drive


In [6]:
#【Cell 2: TokenizerのロードとEOSトークン取得】
# Qwen3-4Bの公式リポジトリからトークナイザを取得し、終端記号を確定させる。
# 2. Tokenizerの初期化とEOSトークンの特定
# 2. Tokenizerの初期化と境界トークンの特定
from transformers import AutoTokenizer

model_id = "Qwen/Qwen3-4B"
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# Qwen3の階層的境界トークン
eos_doc_token = "<|endoftext|>"  # 文書全体の境界
eos_turn_token = "<|im_end|>"    # 対話1ターンの境界

# 検証用に文書境界トークンのIDを取得
eos_doc_id = tokenizer.convert_tokens_to_ids(eos_doc_token)

print(f"文書境界トークン: {eos_doc_token} (ID: {eos_doc_id})")
print(f"ターン境界トークン: {eos_turn_token}")

文書境界トークン: <|endoftext|> (ID: 151643)
ターン境界トークン: <|im_end|>


In [7]:
#【Cell 3: 正規化結合・階層的EOS挿入・シャッフル】
# 対話ログにおけるターンの終了と、テキスト（Note/対話セッション）の終了を明確に区別してマーキングする。
# 3. ファイルの結合、階層的EOS挿入、シャッフル
import json
import random

file_note = base_dir + "note_corpus.jsonl"
file_chat = base_dir + "qwen_sft_dialogues.jsonl"

def load_jsonl(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

note_data = load_jsonl(file_note)
chat_data = load_jsonl(file_chat)

all_texts = []

# Noteコーパスの抽出 (文書終端のみ)
for item in note_data:
    title = item.get("title", "")
    body = item.get("body", "")
    text = f"{title}\n{body}" if title else body
    text_with_eos = text + eos_doc_token
    all_texts.append(text_with_eos)

# 対話ログの抽出 (ターン終端 + 文書終端)
for item in chat_data:
    messages = item.get("messages", [])
    chat_text = ""
    for msg in messages:
        role = msg.get("role", "")
        content = msg.get("content", "")
        chat_text += f"<|im_start|>{role}\n{content}{eos_turn_token}\n"

    # セッションの最後に文書境界トークンを付与
    text_with_eos = chat_text + eos_doc_token
    all_texts.append(text_with_eos)

# シャッフル
random.seed(42)
random.shuffle(all_texts)

print(f"Note documents: {len(note_data)}")
print(f"Chat documents: {len(chat_data)}")
print(f"Total documents: {len(all_texts)}")

Note documents: 657
Chat documents: 1654
Total documents: 2311


In [8]:
#【Cell 4: トークン化・パース検証・2048チャンク出力】
# 文字列として埋め込まれた特殊トークンをIDへと正しくパースさせるため allowed_special="all" を使用する。
# 切り出し後、Unslothへの入力互換性を担保するため、トークン配列を文字列へデコードし ensure_ascii=False で保存する
# 4. トークナイズ、検証、2048チャンキング、Unsloth形式保存
import os

chunk_size = 2048
output_dir = base_dir + "cpt_chunks/"
os.makedirs(output_dir, exist_ok=True)

all_token_ids = []

print("エンコードを実行中...")
for text in all_texts:
    # 手動埋め込みした特殊トークン文字列をIDへ変換させるための指定
    tokens = tokenizer.encode(text, allowed_special="all")
    all_token_ids.extend(tokens)

# EOSトークンのパース検証
actual_eos_count = all_token_ids.count(eos_doc_id)
print(f"期待されるEOS出現回数(文書数): {len(all_texts)}")
print(f"エンコード後の実際のEOS出現回数: {actual_eos_count}")

# 2048トークンごとにスライス
chunks = [all_token_ids[i:i + chunk_size] for i in range(0, len(all_token_ids), chunk_size)]

# 最終チャンクの破棄とトークン数計算
dropped_tokens = 0
if len(chunks[-1]) < chunk_size:
    dropped_tokens = len(chunks[-1])
    chunks = chunks[:-1]

print(f"生成されたチャンク数: {len(chunks)}")
print(f"末尾破棄トークン数: {dropped_tokens}")
print(f"総有効トークン数: {len(chunks) * chunk_size}")

# Unsloth用にデコードし、"text" キーを持つJSONLとして保存
output_file = output_dir + "qwen3_4b_cpt_dataset_2048.jsonl"
with open(output_file, 'w', encoding='utf-8') as f:
    for chunk in chunks:
        # トークンIDの配列を再び文字列（テキストチャンク）へ復元
        decoded_text = tokenizer.decode(chunk)
        record = {"text": decoded_text}
        f.write(json.dumps(record, ensure_ascii=False) + '\n')

print(f"保存完了: {output_file}")


エンコードを実行中...


Token indices sequence length is longer than the specified maximum sequence length for this model (283759 > 131072). Running this sequence through the model will result in indexing errors


期待されるEOS出現回数(文書数): 2311
エンコード後の実際のEOS出現回数: 2311
生成されたチャンク数: 30094
末尾破棄トークン数: 1633
総有効トークン数: 61632512
保存完了: /content/drive/MyDrive/cooking_note/cpt_chunks/qwen3_4b_cpt_dataset_2048.jsonl
